# 🎮 Phase 6 - Task 6.1: Weighted Hybrid Recommender Prototyping

> **Mục tiêu của Notebook:**
> 1. **Data & Multi-Model Integration:** Tích hợp cả 3 nguồn tri thức: Collaborative Filtering (TruncatedSVD), Content-Based (Semantic Embeddings Centroid) và Sentiment Analysis (VADER Compound + Positive Ratio).
> 2. **Score Normalization:** Chuẩn hóa các thang điểm dị biệt (SVD dot-product, Cosine similarity, VADER polarity) về cùng không gian $[0, 1]$.
> 3. **Dynamic Weighted Hybrid Fusion:** Xây dựng cơ chế kết hợp tuyến tính có trọng số $\text{Score} = w_{\text{cf}} \cdot S_{\text{cf}} + w_{\text{cb}} \cdot S_{\text{cb}} + w_{\text{sent}} \cdot S_{\text{sent}}$, tự động chuyển đổi cấu hình khi gặp Cold-Start User.
> 4. **Comparative Analysis & Case Studies:** Đánh giá so sánh trực quan gợi ý giữa Pure CF, Pure Content-Based, Sentiment-Augmented và Balanced Hybrid trên cả người dùng cũ (Warm User) và người dùng mới (Cold-Start User).
> 5. **Weight Sensitivity & Distribution Analysis:** Trực quan hóa tương quan điểm số và độ ổn định của hệ thống.

In [ ]:
import os
import sys
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.append("..")
from src.models.collaborative.matrix_factorization import SVDRecommender
from src.models.content_based.recommender import ContentBasedRecommender

# Thiết lập hiển thị & biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

print("[*] Libraries imported successfully!")

## 1. Tải Dữ liệu Silver/Gold & Các Mô hình Đã Huấn luyện

- **Item Features & Metadata:** 25,612 tựa game.
- **Item Sentiment Profiles:** Thống kê VADER sentiment cho từng game.
- **Pre-trained SVD Model:** TruncatedSVD ($k=64$ latent factors).
- **Content-Based Model:** Sentence-Transformers 384-D item embeddings.

In [ ]:
# 1. Tải Content-Based Recommender (kèm Embeddings và Item Features)
cb_model = ContentBasedRecommender(
    embeddings_path="../data/gold/item_embeddings.npy",
    items_path="../data/silver/item_features.parquet"
)

# 2. Tải SVD Collaborative Filtering Model
svd_model = SVDRecommender.load_model("../models/collaborative/svd_recommender.joblib")

# 3. Tải Item Sentiment Profiles
df_sentiment = pl.read_parquet("../data/silver/item_sentiment.parquet")

# 4. Tải Interactions để lấy lịch sử tương tác người dùng
df_interactions = pl.read_parquet("../data/silver/interactions.parquet")

print(f"[+] Catalog items: {len(cb_model.item_ids):,}")
print(f"[+] SVD Model Users: {len(svd_model.user2idx):,} | Items: {len(svd_model.item2idx):,}")
print(f"[+] Sentiment profiles: {len(df_sentiment):,} items")
print(f"[+] Clean interactions: {len(df_interactions):,}")

## 2. Tiền xử lý & Chuẩn hóa Điểm số (Score Normalization)

Các thành phần trong hệ thống có thang đo khác nhau:
- **SVD Collaborative Score ($S_{\text{cf}}$):** Tích vô hướng vector người dùng và vật phẩm, giá trị thường dao động từ $-2.0$ đến $+6.0$.
- **Content-Based Score ($S_{\text{cb}}$):** Cosine similarity giữa centroid sở thích và item embeddings, giá trị từ $-1.0$ đến $+1.0$ (thường là $0.0 \to 0.8$).
- **Sentiment Score ($S_{\text{sent}}$):** Kết hợp tỷ lệ đánh giá tích cực (`positive_review_ratio` $\in [0, 1]$) và VADER Compound Score (`avg_sentiment_compound` $\in [-1, 1]$).

$$S_{\text{sent}}(i) = 0.6 \cdot \text{positive\_ratio}(i) + 0.4 \cdot \left(\frac{\text{compound}(i) + 1}{2}\right)$$

In [ ]:
# Xây dựng sentiment lookup array đồng bộ theo thứ tự item_ids của catalog
sent_dict = {
    row["parent_asin"]: {
        "pos_ratio": row["positive_review_ratio"],
        "compound": row["avg_sentiment_compound"],
        "reviews": row["review_count"]
    }
    for row in df_sentiment.iter_rows(named=True)
}

item_sentiment_scores = np.zeros(len(cb_model.item_ids), dtype=np.float32)
for i, iid in enumerate(cb_model.item_ids):
    if iid in sent_dict:
        pos = sent_dict[iid]["pos_ratio"]
        comp = sent_dict[iid]["compound"]
        norm_comp = (comp + 1.0) / 2.0  # Scale [-1, 1] -> [0, 1]
        item_sentiment_scores[i] = 0.6 * pos + 0.4 * norm_comp
    else:
        item_sentiment_scores[i] = 0.5  # Neutral default for items without reviews

print(f"[+] Item sentiment score range: min={item_sentiment_scores.min():.4f}, max={item_sentiment_scores.max():.4f}, mean={item_sentiment_scores.mean():.4f}")

## 3. Thuật toán Weighted Hybrid Engine Prototyping

Công thức tính Hybrid Affinity Score:
$$\text{HybridScore}(u, i) = w_{\text{cf}} \cdot \hat{S}_{\text{cf}}(u, i) + w_{\text{cb}} \cdot \hat{S}_{\text{cb}}(u, i) + w_{\text{sent}} \cdot \hat{S}_{\text{sent}}(i)$$

Trong đó:
- $\sum w = 1.0$
- Nếu User là **Cold-Start** (không có vector CF), hệ thống tự động fallback: $w_{\text{cf}} = 0.0, w_{\text{cb}} = 0.8, w_{\text{sent}} = 0.2$.

In [ ]:
def min_max_scale(arr: np.ndarray) -> np.ndarray:
    """Scale array linearly to [0, 1]."""
    min_v = np.min(arr)
    max_v = np.max(arr)
    if max_v > min_v:
        return (arr - min_v) / (max_v - min_v)
    return np.zeros_like(arr)

def get_hybrid_recommendations(
    user_id: str = None,
    liked_item_ids: list = None,
    liked_weights: list = None,
    w_cf: float = 0.50,
    w_cb: float = 0.35,
    w_sent: float = 0.15,
    top_k: int = 10,
    exclude_interacted: bool = True
):
    """
    Compute weighted hybrid recommendations across the entire catalog.
    """
    n_items = len(cb_model.item_ids)
    interacted_asins = set()
    
    # 1. Collaborative Filtering Score Vector
    cf_available = False
    scores_cf = np.zeros(n_items, dtype=np.float32)
    if user_id and user_id in svd_model.user2idx:
        u_idx = svd_model.user2idx[user_id]
        u_vec = svd_model.user_factors[u_idx]
        # TruncatedSVD dot product for all SVD items
        svd_item_scores = np.dot(u_vec, svd_model.item_factors.T)
        # Map to catalog order
        for i, iid in enumerate(cb_model.item_ids):
            if iid in svd_model.item2idx:
                scores_cf[i] = svd_item_scores[svd_model.item2idx[iid]]
            else:
                scores_cf[i] = svd_model.global_mean
        scores_cf = min_max_scale(scores_cf)
        cf_available = True
        
        # Lấy lịch sử tương tác của user
        u_hist = df_interactions.filter(pl.col("user_id") == user_id)["parent_asin"].to_list()
        interacted_asins.update(u_hist)
    
    # 2. Content-Based Score Vector
    scores_cb = np.zeros(n_items, dtype=np.float32)
    if liked_item_ids:
        interacted_asins.update(liked_item_ids)
        valid_indices = [cb_model.item2idx[iid] for iid in liked_item_ids if iid in cb_model.item2idx]
        if valid_indices:
            w = np.array(liked_weights if liked_weights else [1.0] * len(valid_indices), dtype=np.float32)
            item_vecs = cb_model.embeddings[valid_indices]
            user_centroid = np.sum(item_vecs * w.reshape(-1, 1), axis=0)
            norm = np.linalg.norm(user_centroid)
            if norm > 0:
                user_centroid /= norm
            scores_cb = np.dot(cb_model.embeddings, user_centroid)
            scores_cb = min_max_scale(scores_cb)
    elif user_id and cf_available:
        # Lấy top liked items từ interactions của user để tạo profile
        user_top_items = df_interactions.filter(pl.col("user_id") == user_id).sort("rating", descending=True)["parent_asin"].to_list()[:5]
        valid_indices = [cb_model.item2idx[iid] for iid in user_top_items if iid in cb_model.item2idx]
        if valid_indices:
            user_centroid = np.mean(cb_model.embeddings[valid_indices], axis=0)
            norm = np.linalg.norm(user_centroid)
            if norm > 0:
                user_centroid /= norm
            scores_cb = np.dot(cb_model.embeddings, user_centroid)
            scores_cb = min_max_scale(scores_cb)

    # 3. Sentiment Score Vector
    scores_sent = item_sentiment_scores.copy()

    # 4. Adaptive Weights Fallback for Cold-Start
    if not cf_available:
        effective_w_cf = 0.0
        sum_w = w_cb + w_sent
        effective_w_cb = w_cb / sum_w if sum_w > 0 else 0.8
        effective_w_sent = w_sent / sum_w if sum_w > 0 else 0.2
    else:
        total_w = w_cf + w_cb + w_sent
        effective_w_cf = w_cf / total_w
        effective_w_cb = w_cb / total_w
        effective_w_sent = w_sent / total_w

    # 5. Hybrid Linear Fusion
    hybrid_scores = (
        effective_w_cf * scores_cf +
        effective_w_cb * scores_cb +
        effective_w_sent * scores_sent
    )

    # Mask interacted items
    if exclude_interacted and interacted_asins:
        for iid in interacted_asins:
            if iid in cb_model.item2idx:
                hybrid_scores[cb_model.item2idx[iid]] = -np.inf

    # Top-K
    top_idx = np.argpartition(hybrid_scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(-hybrid_scores[top_idx])]

    results = []
    for idx in top_idx:
        iid = cb_model.idx2item[idx]
        results.append({
            "parent_asin": iid,
            "title": cb_model.item_titles.get(iid, "Unknown"),
            "category": cb_model.item_categories.get(iid, "Unknown"),
            "hybrid_score": round(float(hybrid_scores[idx]), 4),
            "cf_score": round(float(scores_cf[idx]), 4) if cf_available else 0.0,
            "cb_score": round(float(scores_cb[idx]), 4),
            "sentiment_score": round(float(scores_sent[idx]), 4),
            "avg_rating": cb_model.item_ratings.get(iid, 0.0),
        })
    
    return results, (effective_w_cf, effective_w_cb, effective_w_sent)

## 4. Case Study 1: Người dùng Hiện hữu (Warm User)

Chọn một người dùng có nhiều tương tác thực tế từ tập dữ liệu và so sánh gợi ý giữa các chiến lược:
1. **Pure CF ($w_{\text{cf}}=1.0$)**
2. **Pure Content-Based ($w_{\text{cb}}=1.0$)**
3. **Balanced Hybrid ($w_{\text{cf}}=0.50, w_{\text{cb}}=0.35, w_{\text{sent}}=0.15$)**

In [ ]:
# Chọn 1 user có tương tác phong phú
top_users = df_interactions.group_by("user_id").agg(pl.len().alias("count")).sort("count", descending=True)
sample_user_id = top_users["user_id"][5]
sample_user_history = df_interactions.filter(pl.col("user_id") == sample_user_id).join(
    cb_model.df_items.select(["parent_asin", "title", "main_category"]),
    on="parent_asin",
    how="left"
)

print(f"👤 Sample User ID: {sample_user_id} (Tổng số đánh giá: {len(sample_user_history)})")
print("\n--- Top 5 Game User Đã Chơi & Đánh Giá ---")
for row in sample_user_history.sort("rating", descending=True).head(5).iter_rows(named=True):
    print(f"⭐ {row['rating']} Sao | [{row['parent_asin']}] {row['title']} ({row.get('main_category', 'N/A')})")

In [ ]:
# Chạy 3 cấu hình gợi ý cho Warm User
recs_cf, _ = get_hybrid_recommendations(user_id=sample_user_id, w_cf=1.0, w_cb=0.0, w_sent=0.0, top_k=5)
recs_cb, _ = get_hybrid_recommendations(user_id=sample_user_id, w_cf=0.0, w_cb=1.0, w_sent=0.0, top_k=5)
recs_hybrid, eff_weights = get_hybrid_recommendations(
    user_id=sample_user_id, w_cf=0.50, w_cb=0.35, w_sent=0.15, top_k=5
)

print("=" * 95)
print(f"🔥 SO SÁNH GỢI Ý CHO WARM USER ({sample_user_id})")
print("=" * 95)

print("\n1️⃣ [Pure CF (SVD 100%)]:")
for i, r in enumerate(recs_cf, 1):
    print(f"   {i}. [{r['parent_asin']}] {r['title'][:45]:<45} | CF: {r['cf_score']:.3f} | Rating: {r['avg_rating']}")

print("\n2️⃣ [Pure Content-Based (Embeddings 100%)]:")
for i, r in enumerate(recs_cb, 1):
    print(f"   {i}. [{r['parent_asin']}] {r['title'][:45]:<45} | CB: {r['cb_score']:.3f} | Rating: {r['avg_rating']}")

print(f"\n3️⃣ [Balanced Hybrid ({eff_weights[0]*100:.0f}% CF + {eff_weights[1]*100:.0f}% CB + {eff_weights[2]*100:.0f}% Sentiment)]:")
for i, r in enumerate(recs_hybrid, 1):
    print(f"   {i}. [{r['parent_asin']}] {r['title'][:45]:<45} | Hybrid: {r['hybrid_score']:.3f} (CF:{r['cf_score']:.2f}, CB:{r['cb_score']:.2f}, Sent:{r['sentiment_score']:.2f}) | Rating: {r['avg_rating']}")

## 5. Case Study 2: Người dùng Mới (Cold-Start Zero-Shot User)

Một người dùng mới đăng ký chỉ cung cấp 2 tựa game yêu thích (ví dụ: Game Thể thao / Đua xe hoặc Game Hành động Nhập vai).
Hệ thống không có lịch sử Collaborative Filtering $\rightarrow$ Cơ chế Adaptive Weights tự động kích hoạt.

In [ ]:
# Giả lập user mới chọn 2 tựa game RPG / Phiêu lưu
cold_liked_items = [cb_model.item_ids[10], cb_model.item_ids[15]]
print("🆕 Cold-Start User sở thích ban đầu:")
for iid in cold_liked_items:
    print(f"   - [{iid}] {cb_model.item_titles[iid]} ({cb_model.item_categories[iid]})")

# Gọi Hybrid Recommender (không có user_id)
recs_cold, eff_cold_weights = get_hybrid_recommendations(
    liked_item_ids=cold_liked_items,
    w_cf=0.50, w_cb=0.35, w_sent=0.15,
    top_k=6
)

print(f"\n🚀 Gợi ý Adaptive Hybrid ({eff_cold_weights[0]*100:.0f}% CF + {eff_cold_weights[1]*100:.0f}% CB + {eff_cold_weights[2]*100:.0f}% Sentiment):")
print("-" * 90)
print(f"{'Rank':<5} | {'Hybrid':<8} | {'CB Sim':<8} | {'Sentiment':<10} | {'Rating':<8} | {'Title'}")
print("-" * 90)
for rank, r in enumerate(recs_cold, 1):
    print(f"{rank:<5} | {r['hybrid_score']:<8.4f} | {r['cb_score']:<8.4f} | {r['sentiment_score']:<10.4f} | {r['avg_rating']:<8.1f} | {r['title']}")

## 6. Phân tích Độ nhạy Trọng số (Weight Sensitivity & Overlap Analysis)

Khảo sát sự thay đổi của danh sách Top-10 khi dịch chuyển trọng số từ thuần CF ($w_{\text{cf}} = 1.0$) sang thuần Content-Based ($w_{\text{cb}} = 1.0$).

In [ ]:
alpha_values = np.linspace(0.0, 1.0, 11)
overlap_with_cf = []
overlap_with_cb = []

top10_pure_cf = {r['parent_asin'] for r in recs_cf}
top10_pure_cb = {r['parent_asin'] for r in recs_cb}

for alpha in alpha_values:
    w_cf_curr = alpha
    w_cb_curr = 1.0 - alpha
    recs_test, _ = get_hybrid_recommendations(
        user_id=sample_user_id,
        w_cf=w_cf_curr,
        w_cb=w_cb_curr,
        w_sent=0.0,
        top_k=5
    )
    current_asins = {r['parent_asin'] for r in recs_test}
    
    # Jaccard / Overlap ratio
    overlap_with_cf.append(len(current_asins.intersection(top10_pure_cf)) / len(top10_pure_cf))
    overlap_with_cb.append(len(current_asins.intersection(top10_pure_cb)) / len(top10_pure_cb))

# Vẽ biểu đồ
plt.figure(figsize=(9, 4.5))
plt.plot(alpha_values, overlap_with_cf, marker='o', linewidth=2, label='Tỷ lệ trùng lặp với Pure CF')
plt.plot(alpha_values, overlap_with_cb, marker='s', linewidth=2, label='Tỷ lệ trùng lặp với Pure CB')
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Điểm cân bằng (50/50)')
plt.title('Weight Sensitivity Analysis: Trọng số CF vs Content-Based', fontsize=13, fontweight='bold')
plt.xlabel('Trọng số Collaborative Filtering ($w_{cf}$)', fontsize=11)
plt.ylabel('Tỷ lệ trùng lặp (Overlap Ratio)', fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## 7. Tổng kết Đánh giá Task 6.1

> ### 💡 Những kết luận quan trọng:
> 1. **Giải quyết triệt để Cold-Start:** Cơ chế trọng số thích ứng (Adaptive Weights) tự động vô hiệu hóa thành phần CF khi không có lịch sử người dùng và phân bổ trọng số cho Content-Based + Sentiment.
> 2. **Cân bằng Serendipity & Relevance:** Pure CF có thể tạo ra các phát hiện thú vị (serendipity) theo cộng đồng nhưng dễ bị ảnh hưởng bởi độ phổ biến; Pure Content-Based đảm bảo độ tương đồng nội dung cao nhưng dễ rơi vào filter bubble; **Hybrid Fusion (50% CF + 35% CB + 15% Sent)** mang lại danh mục gợi ý toàn diện và chất lượng cao nhất.
> 3. **Bộ lọc chất lượng bằng Sentiment:** Thành phần Sentiment giúp ưu tiên các tựa game được cộng đồng đánh giá tích cực thực chất qua văn bản review, loại bỏ các game có rating cao nhưng review tiêu cực/châm biếm.
> 
> **Bước kế tiếp (Task 6.2):** Thử nghiệm trích xuất tín hiệu giải thích lý do gợi ý (Explanation Signals & Sentiment Reasons) trên notebook.